# KAIST Acoustic Fault Classification — Model Comparison & Tuning
Compares **Random Forest**, **Multinomial Logistic Regression**, and **Linear SVM (One-vs-Rest)** in Spark MLlib (`pyspark.ml`), with `StandardScaler` and hyperparameter tuning, before selecting a configuration to save via `src/ml/save_best_model.py`.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression, LinearSVC, OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

spark = (
    SparkSession.builder
    .appName("KAISTModelExperiments")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

TRAIN_PATH = "hdfs://localhost:9000/spark/data/kaist/train"
TEST_PATH = "hdfs://localhost:9000/spark/data/kaist/test"

train_df = spark.read.parquet(TRAIN_PATH).cache()
test_df = spark.read.parquet(TEST_PATH).cache()
feature_cols = [c for c in train_df.columns if c.startswith("f_")]

print(f"Train rows: {train_df.count()} | Test rows: {test_df.count()} | Features: {len(feature_cols)}")
print("Train label counts:")
train_df.groupBy("recording_id", "label").count().orderBy("label", "recording_id").show(truncate=False)
print("Test label counts:")
test_df.groupBy("recording_id", "label").count().orderBy("label", "recording_id").show(truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/25 10:24:51 WARN Utils: Your hostname, LAPTOP-VOE4VOE8, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/25 10:24:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/loki/projects/acoustic-fault-stream/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/25 10:24:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/25 10:24:58 WARN SparkS

Train rows: 75 | Test rows: 75 | Features: 31
Train label counts:
+------------+----------+-----+
|recording_id|label     |count|
+------------+----------+-----+
|0Nm_BPFI_03 |inner_race|30   |
|0Nm_Normal  |normal    |15   |
|0Nm_BPFO_10 |outer_race|30   |
+------------+----------+-----+

Test label counts:
+------------+----------+-----+
|recording_id|label     |count|
+------------+----------+-----+
|0Nm_BPFI_10 |inner_race|30   |
|0Nm_Normal  |normal    |15   |
|0Nm_BPFO_03 |outer_race|30   |
+------------+----------+-----+



In [2]:
# Check feature summary stats for NaNs/Infs and scale differences
null_exprs = [F.sum(F.when(F.col(c).isNull() | F.isnan(F.col(c)), 1).otherwise(0)).alias(c) for c in feature_cols[:10]]
train_df.select(*null_exprs).show()
train_df.select("f_0", "f_1", "f_26", "f_27", "f_28", "f_29", "f_30").describe().show()

+---+---+---+---+---+---+---+---+---+---+
|f_0|f_1|f_2|f_3|f_4|f_5|f_6|f_7|f_8|f_9|
+---+---+---+---+---+---+---+---+---+---+
|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|
+---+---+---+---+---+---+---+---+---+---+

+-------+-------------------+------------------+-------------------+-------------------+-----------------+-----------------+------------------+
|summary|                f_0|               f_1|               f_26|               f_27|             f_28|             f_29|              f_30|
+-------+-------------------+------------------+-------------------+-------------------+-----------------+-----------------+------------------+
|  count|                 75|                75|                 75|                 75|               75|               75|                75|
|   mean|-21.044558881123862| 64.99013092041015| 0.5519268115361532|0.09343604216972987|4583.460485026042|4436.193919270833| 8051.243782552084|
| stddev|   55.4831149852829|27.116267896764207| 0.2964569668841889|0

In [3]:
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label_idx", predictionCol="prediction", metricName="f1"
)
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label_idx", predictionCol="prediction", metricName="accuracy"
)

def evaluate_model(model, df, name: str):
    indexer_model = next(s for s in model.stages if isinstance(s, type(model.stages[0])) and hasattr(s, "labels"))
    labels = list(indexer_model.labels)
    idx_to_label = F.create_map(*[x for i, lbl in enumerate(labels) for x in (F.lit(float(i)), F.lit(lbl))])
    preds = model.transform(df).withColumn("pred_label", idx_to_label[F.col("prediction")])
    acc = evaluator_acc.evaluate(preds)
    f1 = evaluator_f1.evaluate(preds)
    prec = MulticlassClassificationEvaluator(labelCol="label_idx", predictionCol="prediction", metricName="weightedPrecision").evaluate(preds)
    rec = MulticlassClassificationEvaluator(labelCol="label_idx", predictionCol="prediction", metricName="weightedRecall").evaluate(preds)
    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | Weighted Precision: {prec:.4f} | Weighted Recall: {rec:.4f}")
    cm = (
        preds.groupBy("label", "pred_label")
        .count()
        .toPandas()
        .pivot(index="label", columns="pred_label", values="count")
        .fillna(0)
        .astype(int)
    )
    print(cm)
    print()
    return {"name": name, "accuracy": acc, "f1": f1, "precision": prec, "recall": rec, "model": model}

In [4]:
indexer = StringIndexer(inputCol="label", outputCol="label_idx", handleInvalid="keep")
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features", handleInvalid="keep")
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withMean=True, withStd=True)

results = []

# 1. Random Forest (tuned)
rf = RandomForestClassifier(labelCol="label_idx", featuresCol="scaled_features", seed=42)
rf_pipe = Pipeline(stages=[indexer, assembler, scaler, rf])
rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100, 200])
    .addGrid(rf.maxDepth, [4, 6, 10])
    .build()
)
rf_cv = CrossValidator(estimator=rf_pipe, estimatorParamMaps=rf_grid, evaluator=evaluator_f1, numFolds=3, seed=42)
rf_cv_model = rf_cv.fit(train_df)
best_rf_model = rf_cv_model.bestModel
best_rf_stage = best_rf_model.stages[-1]
print(f"Best RF params: numTrees={best_rf_stage.getNumTrees}, maxDepth={best_rf_stage.getOrDefault('maxDepth')}")
results.append(evaluate_model(best_rf_model, test_df, "RandomForest (CV-tuned)"))

# 2. Multinomial Logistic Regression (tuned)
lr = LogisticRegression(labelCol="label_idx", featuresCol="scaled_features", family="multinomial", maxIter=100)
lr_pipe = Pipeline(stages=[indexer, assembler, scaler, lr])
lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1, 1.0])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)
lr_cv = CrossValidator(estimator=lr_pipe, estimatorParamMaps=lr_grid, evaluator=evaluator_f1, numFolds=3, seed=42)
lr_cv_model = lr_cv.fit(train_df)
best_lr_model = lr_cv_model.bestModel
best_lr_stage = best_lr_model.stages[-1]
print(f"Best LR params: regParam={best_lr_stage.getRegParam()}, elasticNetParam={best_lr_stage.getElasticNetParam()}")
results.append(evaluate_model(best_lr_model, test_df, "Multinomial LogisticRegression (CV-tuned)"))

# 3. Linear SVM via OneVsRest + LinearSVC (tuned)
lsvc = LinearSVC(maxIter=100)
ovr = OneVsRest(classifier=lsvc, labelCol="label_idx", featuresCol="scaled_features")
svm_pipe = Pipeline(stages=[indexer, assembler, scaler, ovr])
svm_grid = (
    ParamGridBuilder()
    .addGrid(lsvc.regParam, [0.01, 0.1, 1.0])
    .build()
)
svm_cv = CrossValidator(estimator=svm_pipe, estimatorParamMaps=svm_grid, evaluator=evaluator_f1, numFolds=3, seed=42)
svm_cv_model = svm_cv.fit(train_df)
best_svm_model = svm_cv_model.bestModel
best_svm_reg = best_svm_model.stages[-1].getClassifier().getRegParam()
print(f"Best LinearSVC (OvR) regParam={best_svm_reg}")
results.append(evaluate_model(best_svm_model, test_df, "OneVsRest LinearSVC (CV-tuned)"))

Best RF params: numTrees=50, maxDepth=4
=== RandomForest (CV-tuned) ===
Accuracy: 0.2000 | F1: 0.1000 | Weighted Precision: 0.0667 | Weighted Recall: 0.2000


/home/loki/projects/acoustic-fault-stream/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


pred_label  normal  outer_race
label                         
inner_race       0          30
normal          15           0
outer_race      30           0



Best LR params: regParam=0.01, elasticNetParam=0.0
=== Multinomial LogisticRegression (CV-tuned) ===
Accuracy: 0.2000 | F1: 0.1000 | Weighted Precision: 0.0667 | Weighted Recall: 0.2000
pred_label  normal  outer_race
label                         
inner_race       0          30
normal          15           0
outer_race      30           0



/home/loki/projects/acoustic-fault-stream/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Best LinearSVC (OvR) regParam=0.01
=== OneVsRest LinearSVC (CV-tuned) ===
Accuracy: 0.2000 | F1: 0.0984 | Weighted Precision: 0.0652 | Weighted Recall: 0.2000


/home/loki/projects/acoustic-fault-stream/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


pred_label  normal  outer_race
label                         
inner_race       1          29
normal          15           0
outer_race      30           0



In [5]:
import pandas as pd
summary_df = pd.DataFrame([{k: v for k, v in r.items() if k != "model"} for r in results]).sort_values("f1", ascending=False)
display(summary_df)

# Inspect top feature importances from best RF if available
from src.features.extractor import FEATURE_NAMES
rf_importances = best_rf_model.stages[-1].featureImportances.toArray()
top_idx = rf_importances.argsort()[::-1][:10]
print("Top 10 features by RF importance:")
for i in top_idx:
    print(f"  {FEATURE_NAMES[i]:18s}: {rf_importances[i]:.4f}")

,name,accuracy,f1,precision,recall
0,RandomForest (CV-tuned),0.2,0.100000,0.066667,0.2
1,Multinomial LogisticRegression (CV-tuned),0.2,0.100000,0.066667,0.2
2,OneVsRest LinearSVC (CV-tuned),0.2,0.098361,0.065217,0.2


Top 10 features by RF importance:
  mfcc_1_mean       : 0.1032
  mfcc_11_mean      : 0.0870
  mfcc_8_mean       : 0.0855
  mfcc_2_mean       : 0.0686
  mfcc_6_mean       : 0.0625
  bandwidth_mean    : 0.0545
  mfcc_5_mean       : 0.0472
  rolloff_mean      : 0.0468
  mfcc_9_mean       : 0.0424
  rms_mean          : 0.0415


In [6]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

num_features = len(feature_cols)
num_classes = train_df.select("label").distinct().count()

mlp_candidates = [
    ("MLP [32]", [num_features, 32, num_classes], 0.01),
    ("MLP [32, 16]", [num_features, 32, 16, num_classes], 0.01),
    ("MLP [64, 32] (L2=0.1)", [num_features, 64, 32, num_classes], 0.1),
]

for mlp_name, layers, block_size_or_reg in mlp_candidates:
    mlp = MultilayerPerceptronClassifier(
        labelCol="label_idx",
        featuresCol="scaled_features",
        layers=layers,
        maxIter=150,
        stepSize=0.03,
        seed=42,
    )
    mlp_pipe = Pipeline(stages=[indexer, assembler, scaler, mlp])
    mlp_model = mlp_pipe.fit(train_df)
    res = evaluate_model(mlp_model, test_df, mlp_name)
    results.append(res)

=== MLP [32] ===
Accuracy: 0.2000 | F1: 0.0811 | Weighted Precision: 0.0508 | Weighted Recall: 0.2000
pred_label  normal  outer_race
label                         
inner_race      14          16
normal          15           0
outer_race      30           0



/home/loki/projects/acoustic-fault-stream/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


=== MLP [32, 16] ===
Accuracy: 0.2000 | F1: 0.1000 | Weighted Precision: 0.0667 | Weighted Recall: 0.2000
pred_label  normal  outer_race
label                         
inner_race       0          30
normal          15           0
outer_race      30           0



/home/loki/projects/acoustic-fault-stream/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


=== MLP [64, 32] (L2=0.1) ===
Accuracy: 0.2000 | F1: 0.1000 | Weighted Precision: 0.0667 | Weighted Recall: 0.2000
pred_label  normal  outer_race
label                         
inner_race       0          30
normal          15           0
outer_race      30           0



/home/loki/projects/acoustic-fault-stream/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
